# Evaluate scoped Loom memory retrieval with Braintrust

Measure whether direct application-controlled searches return the expected synthetic memory first. Three cases exercise a schedule, a retry policy, and an access review. No model selects calls or judges answers. This is a component check, not an end-to-end agent performance benchmark.

Read [the setup, disclosure, and cleanup instructions](README.md) before running. Install `requirements.txt` in your own environment and set the five environment variables described there. No live run or performance result is included in this notebook.

In [ ]:
from pathlib import Path
from uuid import uuid4

from braintrust import EvalAsync, Score
from braintrust.git_fields import GitMetadataSettings

from loom_retrieval import Config, LoomHTTP, cleanup, load_run, retrieve, retrieval_score, seed

client = LoomHTTP(Config.from_env())
journal = Path("loom-fixture-" + uuid4().hex + ".jsonl")

## Create three fixtures explicitly

This cell writes three synthetic memories with a one-day expiry. The request intent and ID are persisted before each write, then each acknowledged identity/key/version receipt is retained in the local journal before another write. Run it once; keep the journal out of version control. A partial seed stops without automatic cleanup. Use the cleanup cell to retire acknowledged versions. An interrupted write without a receipt needs investigation by request ID, not a guessed deletion.

In [ ]:
fixture_run = seed(client, journal)

## Evaluate read-only retrieval

Run after the server has indexed the fixtures. No polling or retries are hidden in the evaluation. `expected_key_at_1` is null for refused, invalid, or partial responses. `response_complete` makes those skipped cases visible. Empty results and ordinary abstentions score zero on retrieval. Read both scores together.

This cell awaits EvalAsync before returning, so the next cleanup cell cannot race the evaluation.

Only synthetic inputs, expected keys, verified fixture result keys, and bounded receipt metadata are uploaded. Request latency excludes fixture creation and Braintrust upload. Returned body-token estimates are not provider billing; full provider usage is unavailable.

In [ ]:
fixture_run, receipts = load_run(client, journal)

def task(input):
    return retrieve(client, fixture_run, input)

def ExpectedKeyAt1(output, expected):
    return Score(**retrieval_score(output, expected))

def ResponseComplete(output):
    return Score(
        name="response_complete",
        score=float(output["status"] in {"ok", "abstained"}),
        metadata={"status": output["status"]},
    )

result = await EvalAsync(
    name="Loom memory retrieval",
    experiment_name=fixture_run["tag"],
    data=fixture_run["cases"],
    task=task,
    scores=[ExpectedKeyAt1, ResponseComplete],
    max_concurrency=1,
    enable_cache=False,
    is_public=False,
    git_metadata_settings=GitMetadataSettings(collect="none"),
    metadata={"invocation": "application_direct", "synthetic_cases": 3},
)

## Clean up acknowledged versions explicitly

Run this cell even if seeding or evaluation failed. It only requests deletion of acknowledged fixture versions under their exact identity. Changed versions remain untouched. Cleanup stops at the first refusal or uncertain request outcome; remaining receipts stay pending. Inspect the returned results and local journal before any explicit rerun. Cleanup is separate so it cannot mask the evaluation error. Deletion follows the server tombstone lifecycle; it does not purge a namespace.

In [ ]:
cleanup_results = cleanup(client, journal)
cleanup_results

## Interpret the scope

Three synthetic cases check integration behavior and retrieval receipts. They do not establish a coding-task success rate, model adoption rate, token savings, cost savings, or a comparison against an agent without Loom. Use a reviewed, held-out dataset and account for failures before making application claims.

See the [contract sources and license](README.md#sources-and-license).